# TriForce Reproduction Experiment

Reproduce TriForce (arXiv:2404.11912) experiments on RunPod A100 80GB.

**Steps:**
1. Check GPU
2. Clone project repo
3. Install dependencies
4. Clone TriForce + prepare data
5. Download models
6. Sanity check (small prefill)
7. Full experiment
8. Results

## 1. GPU Check

In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 2. Clone Project Repository

**Option A:** If you uploaded the repo as a zip, skip this cell.  
**Option B:** Clone from GitHub (fill in your repo URL).

In [ ]:
import os

# ---- CONFIGURE THIS ----
GITHUB_REPO = "YOUR_USERNAME/triforce-experiment"  # <-- Replace with your repo
# ---- END CONFIG ----

PROJECT_DIR = "/workspace/triforce-experiment"

if os.path.isdir(PROJECT_DIR):
    print(f"[INFO] Project already exists at {PROJECT_DIR}")
else:
    !git clone https://github.com/{GITHUB_REPO}.git {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"[INFO] Working directory: {os.getcwd()}")

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt

In [ ]:
# flash-attn requires separate install with --no-build-isolation
# This may take 5-10 minutes to compile
!pip install flash-attn==2.5.7 --no-build-isolation

In [ ]:
# Verify key packages
import torch
import transformers

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

try:
    import flash_attn
    print(f"flash-attn: {flash_attn.__version__}")
except ImportError:
    print("[WARN] flash-attn not installed!")

## 4. Clone TriForce + Prepare Data

In [ ]:
!bash scripts/clone_triforce.sh

# Also install any deps from TriForce's own requirements.txt
!pip install -q -r vendor/TriForce/requirements.txt

In [ ]:
!python scripts/prepare_data.py

## 5. Download Models

Downloads:
- `NousResearch/Yarn-Llama-2-7b-128k` (~14GB)
- `JackFram/llama-68m` (~260MB)

In [ ]:
!bash scripts/download_models.sh

## 6. Sanity Check (Small Prefill)

Quick test with a smaller prefill to verify everything works before the full run.

In [ ]:
%%time

import os
os.chdir("/workspace/triforce-experiment/vendor/TriForce")
print(f"[INFO] Working directory: {os.getcwd()}")
print(f"[INFO] Running sanity check with prefill=4096, dataset=one-shot ...")

# Sanity check: small prefill, single sample
!python test/on_chip.py \
    --prefill 4096 \
    --gen_len 32 \
    --budget 4096 \
    --chunk_size 8 \
    --draft_cache_budget 256 \
    --gamma 6 \
    --top_p 0.9 \
    --temp 0.6 \
    --dataset one-shot

## 7. Full Experiment

Run the full TriForce experiment with official parameters:
- `prefill=124928` (~122K tokens)
- `gen_len=256`
- `budget=4096`, `chunk_size=8`, `gamma=6`

**Expected runtime:** ~30-60 minutes depending on data loading.

In [ ]:
%%time

import os
os.chdir("/workspace/triforce-experiment/vendor/TriForce")

RESULTS_DIR = "/workspace/triforce-experiment/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*60)
print("  TriForce Full Experiment")
print("  prefill=124928, gen_len=256, budget=4096, gamma=6")
print("  dataset=128k (all PG-19 samples)")
print("="*60)

# Official on-chip experiment:
# --target defaults to llama-7B-128K (hardcoded to NousResearch/Yarn-Llama-2-7b-128k)
# --draft defaults to llama-68M (hardcoded to JackFram/llama-68m)
# --dataset 128k uses all samples in data/pg19/
!python test/on_chip.py \
    --prefill 124928 \
    --gen_len 256 \
    --budget 4096 \
    --chunk_size 8 \
    --draft_cache_budget 256 \
    --gamma 6 \
    --top_p 0.9 \
    --temp 0.6 \
    --dataset 128k \
    2>&1 | tee {RESULTS_DIR}/triforce_full_run.log

## 8. Results Collection

In [ ]:
import os

RESULTS_DIR = "/workspace/triforce-experiment/results"
log_file = os.path.join(RESULTS_DIR, "triforce_full_run.log")

if os.path.exists(log_file):
    print("=" * 60)
    print("  EXPERIMENT LOG (last 50 lines)")
    print("=" * 60)
    with open(log_file) as f:
        lines = f.readlines()
    for line in lines[-50:]:
        print(line, end="")
else:
    print("[WARN] No log file found. Run cell 7 first.")

In [ ]:
# Expected results from the paper for comparison:
print("=" * 60)
print("  Expected Results (from TriForce paper)")
print("=" * 60)
print(f"{'Method':<25} {'Latency (ms/tok)':<20} {'Speedup':<10}")
print("-" * 55)
print(f"{'Autoregressive':<25} {'~46-48':<20} {'1.0x':<10}")
print(f"{'TriForce':<25} {'~21-22':<20} {'~2.1-2.2x':<10}")
print()
print("Average acceptance rate: ~0.85-0.90")
print()
print("Compare your results above with these expected values.")

In [ ]:
# GPU memory usage after experiment
!nvidia-smi